# Figures and presentation tables

In [ ]:
# Parameters

project_root = "../"
data_dir = "../data"

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from dataclasses import replace

import matplotlib.pyplot as plt
import pandas as pd
import algo_vs_doc.analysis as analysis
import algo_vs_doc.plots as plots
from IPython.display import Markdown, SVG, display
from algo_vs_doc import (
    actionable_ai,
    cohorts,
    low_risk_event_tables,
    prediction_data,
    specialties,
    subgroup_tables,
    table_presentation,
    ward_descriptives,
    word_tables,
)
from algo_vs_doc.scoring.final_assembly import apply_enrolment_funnel

pd.set_option("display.max_columns", None)

outputs_folder = os.path.join(project_root, "results", "summaries")
plot_data_folder = os.path.join(project_root, "results", "plot_data")
for name in cohorts.COHORT_ORDER:
    os.makedirs(os.path.join(plot_data_folder, name), exist_ok=True)
figure_dirs = {
    name: os.path.join(project_root, "figures", name) for name in cohorts.COHORT_ORDER
}
for path in figure_dirs.values():
    os.makedirs(path, exist_ok=True)

report_cohort = "Combined"
formatted_tables_dir = os.path.join(project_root, "tables")
os.makedirs(formatted_tables_dir, exist_ok=True)
cohort_labels = {"Combined": "Both Hospitals", "VUmc": "VUmc", "LUMC": "LUMC"}


def publication_table_frame(frame):
    result = frame.rename(columns={"Combined": "Both Hospitals"}).copy()
    for column in ("Cohort", "cohort"):
        if column in result:
            result[column] = result[column].astype(object).replace(cohort_labels)
    return result

In [ ]:
loaded_sites = prediction_data.load_cohorts(data_dir, include_combined=False)
enrolled_sites = {
    name: apply_enrolment_funnel(frame)[0] for name, frame in loaded_sites.items()
}
loaded_cohorts = {
    "VUmc": enrolled_sites["VUmc"],
    "LUMC": enrolled_sites["LUMC"],
    "Combined": cohorts.combine(enrolled_sites.values()),
}
analysis_cohorts = {}
for name in cohorts.COHORT_ORDER:
    frame = loaded_cohorts[name].copy()
    frame["patient_cluster_id"] = (
        frame["hospital"].astype(str) + "::" + frame["patient_id"].astype(str)
    )
    frame["doc_prediction_cat_score"] = frame["doc_prediction_cat"].map(
        {"low": 0.0, "medium": 1.0, "high": 2.0}
    )
    frame = analysis.combine_predictions(
        frame,
        "doc_prediction",
        "algo_prediction_at_doc_prediction",
        "outcome",
        "algo_and_doc_prediction",
        group_column="patient_cluster_id",
    )
    frame = analysis.combine_predictions(
        frame,
        "doc_prediction_cat",
        "algo_prediction_at_doc_prediction",
        "outcome",
        "algo_and_doc_cat_prediction",
        group_column="patient_cluster_id",
        prediction1_is_ordinal=True,
    )
    analysis_cohorts[name] = frame

## Prediction distributions

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    display(Markdown(f"### {cohort_labels[cohort]}: Physician risk categories"))
    plots.barplot_by_outcome(
        data=predictions_and_outcomes,
        variable="doc_prediction_cat_score",
        output_directory=figure_dir,
    )
    plt.show()

    correlations = pd.read_csv(
        os.path.join(outputs_folder, cohort, "correlation_table.csv"),
        index_col=0,
    )
    display(Markdown(f"### {cohort_labels[cohort]}: Physician and model predictions"))
    plots.scatter_plot(
        data=predictions_and_outcomes,
        xs=["doc_prediction", "doc_prediction", "algo_prediction_at_doc_prediction"],
        ys=[
            "algo_prediction_at_doc_prediction",
            "algo_prediction_at_discharge",
            "algo_prediction_at_discharge",
        ],
        correlation=[
            correlations.iloc[0, 2],
            correlations.iloc[1, 2],
            correlations.iloc[1, 0],
        ],
        output_directory=figure_dir,
    )
    plt.show()

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    display(
        Markdown(f"### {cohort_labels[cohort]}: Prediction distributions by outcome")
    )
    plots.histogram_by_outcome(
        data=predictions_and_outcomes,
        variables=[
            "doc_prediction",
            "algo_prediction_at_doc_prediction",
            "algo_prediction_at_discharge",
        ],
        output_directory=figure_dir,
    )
    plt.show()

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    display(
        Markdown(f"### {cohort_labels[cohort]}: Physician predictions by risk category")
    )
    plots.stratified_histogram(
        predictions_and_outcomes,
        "doc_prediction",
        "doc_prediction_cat",
        labels=["Low Risk", "Moderate Risk", "High Risk"],
        plotname="physician_model_prediction",
        title="Physician Predictions by Risk category",
        output_directory=figure_dir,
    )
    plt.show()

## AUROC contrasts

The forest plot is intended for the main results. The complete bootstrap distributions are a supplementary diagnostic display.

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    auroc_contrasts = pd.read_csv(
        os.path.join(outputs_folder, cohort, "auroc_contrasts.csv")
    )
    auroc_samples = pd.read_csv(
        os.path.join(plot_data_folder, cohort, "auroc_contrast_bootstraps.csv")
    )
    display(Markdown(f"### {cohort_labels[cohort]}: AUROC contrasts"))
    forest_figure = plots.performance_contrast_forest_plot(
        auroc_contrasts,
        output_directory=figure_dir,
        formal_comparisons=cohort == "Combined",
        show_adjusted_p_values=cohort == "Combined",
    )
    display(forest_figure)
    plt.close(forest_figure)
    matrix_figure = plots.performance_contrast_matrix_plot(
        auroc_contrasts,
        auroc_samples,
        output_directory=figure_dir,
    )
    display(matrix_figure)
    plt.close(matrix_figure)

## Exploratory AUPRC comparison matrix
Requires the additional paired contrasts cell in 4-analysis.ipynb.

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    auprc_contrasts = pd.read_csv(
        os.path.join(outputs_folder, cohort, "auprc_contrasts.csv")
    )
    auprc_samples = pd.read_csv(
        os.path.join(plot_data_folder, cohort, "auprc_contrast_bootstraps.csv")
    )
    display(Markdown(f"### {cohort_labels[cohort]}: AUPRC contrasts"))
    forest_figure = plots.performance_contrast_forest_plot(
        auprc_contrasts,
        output_directory=figure_dir,
        formal_comparisons=cohort == "Combined",
        show_adjusted_p_values=cohort == "Combined",
    )
    display(forest_figure)
    plt.close(forest_figure)
    matrix_figure = plots.performance_contrast_matrix_plot(
        auprc_contrasts,
        auprc_samples,
        output_directory=figure_dir,
    )
    display(matrix_figure)
    plt.close(matrix_figure)

## Calibration

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    calibration_curves = pd.read_csv(
        os.path.join(plot_data_folder, cohort, "calibration_curves.csv")
    )
    calibration_deciles = pd.read_csv(
        os.path.join(plot_data_folder, cohort, "calibration_deciles.csv")
    )
    calibration_histograms = pd.read_csv(
        os.path.join(plot_data_folder, cohort, "calibration_histograms.csv")
    )
    display(Markdown(f"### {cohort_labels[cohort]}: Calibration"))
    plots.plot_calibration(
        calibration_curves,
        calibration_deciles,
        calibration_histograms,
        output_directory=figure_dir,
    )
    plt.show()

## Prediction timing

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    time_dif_data = pd.read_csv(
        os.path.join(outputs_folder, cohort, "prediction_timing_sensitivity.csv")
    )
    display(Markdown(f"### {cohort_labels[cohort]}: Prediction timing sensitivity"))
    plots.plot_auc_at_max_time_difference(
        time_dif_data=time_dif_data,
        prediction_columns={
            "Physician (Percentage)": "doc_prediction",
            "Physician (Category)": "doc_prediction_cat_score",
            "Model": "algo_prediction_at_doc_prediction",
        },
        output_directory=figure_dir,
    )
    plt.show()

## Physician confidence

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    display(
        Markdown(f"### {cohort_labels[cohort]}: Physician predictions by confidence")
    )
    plots.stratified_histogram(
        predictions_and_outcomes,
        "doc_prediction",
        "confidence",
        labels=["High Confidence", "Medium/Low Confidence"],
        plotname="confidence_physician_prediction",
        title="Physician Predictions by self-reported Confidence",
        output_directory=figure_dir,
    )
    plt.show()

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    display(
        Markdown(f"### {cohort_labels[cohort]}: Physician confidence by risk category")
    )
    plots.barplot_confidence_cat_prediction(
        predictions_and_outcomes,
        "doc_prediction_cat_score",
        output_directory=figure_dir,
    )
    plt.show()

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    high_confidence = pd.read_csv(
        os.path.join(outputs_folder, cohort, "performance_table_high_confidence.csv"),
        index_col=0,
    )
    lower_confidence = pd.read_csv(
        os.path.join(
            outputs_folder, cohort, "performance_table_medium_low_confidence.csv"
        ),
        index_col=0,
    )
    display(
        Markdown(f"### {cohort_labels[cohort]}: Performance by physician confidence")
    )
    plots.plot_performance_by_confidence(
        lower_confidence, high_confidence, output_directory=figure_dir
    )
    plt.show()

## Feature importance coverage

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    coverage_data = pd.read_csv(
        os.path.join(outputs_folder, cohort, "feature_coverage.csv")
    )
    display(Markdown(f"### {cohort_labels[cohort]}: Feature importance coverage"))
    plots.plot_coverage_at_max_features(coverage_data, output_directory=figure_dir)
    plt.show()

## Discrimination curves

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    display(Markdown(f"### {cohort_labels[cohort]}: AUROC discrimination curves"))
    plots.plot_combined_curves(
        predictions=predictions_and_outcomes,
        outcome="outcome",
        prediction_columns={
            "Physician (Percentage)": "doc_prediction",
            "Physician (Category)": "doc_prediction_cat_score",
            "Model": "algo_prediction_at_doc_prediction",
            "Combined Model and\nPhysician (Percentage)": "algo_and_doc_prediction",
            "Combined Model and\nPhysician (Category)": "algo_and_doc_cat_prediction",
        },
        curve="roc",
        output_directory=figure_dir,
    )
    plt.show()
    display(Markdown(f"### {cohort_labels[cohort]}: AUPRC discrimination curves"))
    plots.plot_combined_curves(
        predictions=predictions_and_outcomes,
        outcome="outcome",
        prediction_columns={
            "Physician (Percentage)": "doc_prediction",
            "Physician (Category)": "doc_prediction_cat_score",
            "Model": "algo_prediction_at_doc_prediction",
            "Combined Model and\nPhysician (Percentage)": "algo_and_doc_prediction",
            "Combined Model and\nPhysician (Category)": "algo_and_doc_cat_prediction",
        },
        curve="prc",
        output_directory=figure_dir,
    )
    plt.show()

## From prediction to possible action

This operational call-out connects fixed alert capacity to possible context-specific actions. It does not imply that an intervention was evaluated.

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    figure_dir = figure_dirs[cohort]
    submission_figure_dir = os.path.join(
        project_root, "results", "submission_figures", cohort
    )
    os.makedirs(submission_figure_dir, exist_ok=True)
    configured_thresholds = pd.read_csv(
        os.path.join(outputs_folder, "configured_threshold_performance.csv")
    )
    display(
        Markdown(f"### {cohort_labels[cohort]}: From prediction to possible action")
    )
    actionable_ai_path = actionable_ai.actionable_ai_svg(
        configured_thresholds,
        os.path.join(figure_dir, "actionable_ai.svg"),
        cohort=cohort,
        pdf_path=os.path.join(submission_figure_dir, "actionable_ai.pdf"),
    )
    display(SVG(filename=str(actionable_ai_path)))

## Editable Word tables

Word tables are rendered from machine-readable CSV outputs without recalculating statistics.

In [ ]:
def read_cohort_outputs(filename):
    paths = [
        (name, os.path.join(outputs_folder, name, filename))
        for name in cohorts.COHORT_ORDER
    ]
    available = [
        pd.read_csv(path).assign(cohort=name)
        for name, path in paths
        if os.path.exists(path)
    ]
    if not available:
        raise FileNotFoundError(f"No cohort output found for {filename}")
    return pd.concat(available, ignore_index=True)


table_1 = pd.read_csv(
    os.path.join(outputs_folder, report_cohort, "baseline_characteristics.csv")
)
table_1 = publication_table_frame(table_1)
table_1_sections = (
    "ICU discharges, n",
    "Age, mean (SD)",
    "Female, n (%)",
    "Admission type, n (%)",
    "Admitting specialty, n (%)",
    "Discharge care setting, n (%)",
    "Length of Stay, days, median (IQR)",
    "Life support during ICU stay, n (%)",
    "Severity scores, median (IQR)",
    "Outcome within 7 days, n (%)",
)
word_tables.write_word_tables(
    [
        word_tables.WordTable(
            "Table 1. Descriptive statistics for ICU discharges",
            table_1,
            notes=(
                "Values are n (%), mean (SD), or median (IQR), as indicated.",
                "The composite outcome comprised ICU readmission or death; the final outcome row reports discharges involving both components.",
                "Outcome percentages use ICU discharges as the denominator, except the patient outcome row, which uses distinct patients.",
                "High-dependency care includes cardiac care, neurological high-care, and medium-care units.",
                "All included discharges were to wards; discharges not identified as high-dependency care were classified as general-ward discharges.",
                "Abbreviations: ICU, intensive care unit; IQR, interquartile range; NEWS, National Early Warning Score; SD, standard deviation; SOFA, Sequential Organ Failure Assessment.",
            ),
            section_starts=table_1_sections,
            indented_rows=(
                "Medical admission",
                "Elective surgical admission",
                "Emergency surgical admission",
                "Unknown",
                "Surgical",
                "Medical",
                "General ward",
                "High-dependency care",
                "Ventilation",
                "Vasopressors or inotropes",
                "Renal-replacement therapy",
                "SOFA at ICU admission",
                "Maximum SOFA during ICU stay",
                "SOFA at ICU discharge",
                "NEWS at ICU discharge",
                "Composite outcome, ICU discharges",
                "Composite outcome, patients",
                "ICU readmission",
                "Death",
                "ICU readmission and death",
            ),
            row_indent_levels={
                label: 2
                for category, label in specialties.SPECIALTY_LABELS.items()
                if category != "unspecified"
            },
        )
    ],
    os.path.join(
        formatted_tables_dir, "Table 1. Descriptive statistics for ICU discharges.docx"
    ),
)

confidence_performance = table_presentation.confidence_performance(
    pd.concat(
        [
            read_cohort_outputs("performance_table_high_confidence.csv").assign(
                confidence="High"
            ),
            read_cohort_outputs("performance_table_medium_low_confidence.csv").assign(
                confidence="Medium or low"
            ),
        ],
        ignore_index=True,
    )
)
factor_categories = table_presentation.factor_categories(
    pd.concat(
        [
            read_cohort_outputs("doc_factor_categories.csv").assign(
                prediction_source="Physician"
            ),
            read_cohort_outputs("algo_factor_categories.csv").assign(
                prediction_source="Model"
            ),
        ],
        ignore_index=True,
    )
)
physician_factor_mapping = pd.read_csv(
    os.path.join(outputs_folder, report_cohort, "physician_factor_mapping.csv")
).rename(
    columns={
        "category": "Category",
        "physician_factor": "Physician factor",
        "VUmc": "VUmc, n",
        "LUMC": "LUMC, n",
    }
)
model_feature_mapping = pd.read_csv(
    os.path.join(outputs_folder, report_cohort, "model_feature_mapping.csv")
).rename(
    columns={
        "category": "Category",
        "model_feature": "Model feature",
        "VUmc": "VUmc, n",
        "LUMC": "LUMC, n",
    }
)
performance = table_presentation.performance(
    read_cohort_outputs("performance_table.csv")
)
contrasts = table_presentation.performance_contrasts(
    pd.concat(
        [
            read_cohort_outputs("auroc_contrasts.csv"),
            read_cohort_outputs("auprc_contrasts.csv"),
        ],
        ignore_index=True,
    )
)
fixed_selection_rates = table_presentation.fixed_selection_rates(
    pd.read_csv(os.path.join(outputs_folder, "fixed_alert_rate_performance.csv"))
)
risk_thresholds = table_presentation.risk_thresholds(
    pd.read_csv(os.path.join(outputs_folder, "configured_threshold_performance.csv"))
)


supplemental_tables = [
    word_tables.WordTable(
        "eTable 1. Mapping of physician factors to feature categories",
        physician_factor_mapping,
        cell_margin_inches=0.03,
        header_fill="1F77B4",
        header_font_color="FFFFFF",
        group_column=["Category", "Physician factor"],
        group_fill="1F77B4",
        group_font_color="FFFFFF",
        merge_group_cells=True,
        notes=(
            "At LUMC, contributing factors were collected as predefined feature categories; individual factors were therefore not available. At VUmc, individually reported factors were mapped to the corresponding feature categories.",
        ),
    ),
    word_tables.WordTable(
        "eTable 2. Mapping of model features to feature categories",
        model_feature_mapping,
        cell_margin_inches=0.03,
        header_fill="2CA02C",
        header_font_color="FFFFFF",
        group_column=["Category", "Model feature"],
        group_fill="2CA02C",
        group_font_color="FFFFFF",
        merge_group_cells=True,
        notes=(
            "For each prediction, the 10 features with the largest absolute SHAP values were included in the mapping.",
        ),
    ),
    word_tables.WordTable(
        "eTable 6. Predictive performance of physicians, the AI model, and combined model-physician predictions",
        performance,
        merge_repeated_columns=("Cohort",),
        notes=(
            "Values in square brackets are patient-cluster bootstrap 95% confidence intervals.",
        ),
    ),
    word_tables.WordTable(
        "eTable 7. Pair-wise comparison of the discriminatory performance of physicians, the AI model and combined model-physician predictions",
        contrasts,
        merge_repeated_columns=("Analysis",),
        header_groups=(
            ("Both Hospitals", ("Both Hospitals AUROC", "Both Hospitals AUPRC")),
            ("VUmc", ("VUmc AUROC", "VUmc AUPRC")),
            ("LUMC", ("LUMC AUROC", "LUMC AUPRC")),
        ),
        group_column=["Analysis", "Agent 1", "Agent 2"],
        header_labels=(
            "Analysis",
            "Agent 1",
            "Agent 2",
            "\u0394AUROC\n(95% CI)",
            "\u0394AUPRC\n(95% CI)",
            "\u0394AUROC\n(95% CI)",
            "\u0394AUPRC\n(95% CI)",
            "\u0394AUROC\n(95% CI)",
            "\u0394AUPRC\n(95% CI)",
        ),
        landscape=True,
        notes=(
            "For model-versus-physician comparisons, \u0394AUROC is the primary analysis and \u0394AUPRC is secondary. P values for these comparisons are Holm adjusted within each metric; p values are not reported for exploratory comparisons.",
        ),
    ),
    word_tables.WordTable(
        "eTable 8. Predictive performance (AUROC) for different physician confidence levels",
        confidence_performance,
        merge_repeated_columns=("Cohort", "Confidence"),
        group_column=["Confidence", "Prediction source"],
        notes=(
            "Values in square brackets are patient-cluster bootstrap 95% confidence intervals.",
        ),
    ),
    word_tables.WordTable(
        "eTable 9. Feature categories contributing to the prediction of physicians and the model",
        factor_categories,
        header_groups=(
            ("Both Hospitals", ("Both Hospitals Physician", "Both Hospitals Model")),
            ("VUmc", ("VUmc Physician", "VUmc Model")),
            ("LUMC", ("LUMC Physician", "LUMC Model")),
        ),
        header_labels=(
            "Feature category",
            "Physician",
            "Model",
            "Physician",
            "Model",
            "Physician",
            "Model",
        ),
    ),
    word_tables.WordTable(
        "eTable 10A. Model operating characteristics at fixed selection rates",
        fixed_selection_rates,
        merge_repeated_columns=("Cohort", "Prediction source"),
        notes=(
            "Values in square brackets are patient-cluster bootstrap 95% confidence intervals.",
            "The selection rate is the prespecified proportion of eligible ICU discharges with the highest model-predicted risks that are designated positive for further assessment; the corresponding risk cutoff is determined separately within each cohort.",
        ),
    ),
    word_tables.WordTable(
        "eTable 10B. Model operating characteristics at risk thresholds",
        risk_thresholds,
        merge_repeated_columns=("Cohort",),
        notes=(
            "Values in square brackets are patient-cluster bootstrap 95% confidence intervals.",
            "The Youden-index threshold maximizes sensitivity plus specificity minus one within each cohort. It was derived and evaluated in the same data and is not a validated clinical threshold.",
        ),
    ),
]


ward_table = pd.read_csv(
    os.path.join(outputs_folder, report_cohort, "event_characteristics_by_outcome.csv"),
    keep_default_na=False,
)
supplemental_tables.append(ward_descriptives.ward_word_table(ward_table))


subgroup_table = pd.read_csv(
    os.path.join(outputs_folder, report_cohort, "clinical_subgroup_table.csv"),
    keep_default_na=False,
)
supplemental_tables.append(subgroup_tables.subgroup_word_table(subgroup_table))


individual_case_dir = os.path.join(project_root, "results", "individual_cases")
individual_entries = pd.read_csv(
    os.path.join(individual_case_dir, "low_risk_event_entries.csv"),
    keep_default_na=False,
)
annotation_path = os.path.join(
    individual_case_dir, "low_risk_event_diagnosis_annotations.csv"
)
diagnosis_annotations = low_risk_event_tables.update_annotation_template(
    individual_entries, annotation_path
)
individual_entries = low_risk_event_tables.add_manual_annotations(
    individual_entries, diagnosis_annotations
)
supplemental_tables.extend(
    low_risk_event_tables.individual_event_word_tables(individual_entries)
)

availability_characteristics = (
    table_presentation.prediction_availability_characteristics(
        pd.read_csv(
            os.path.join(
                outputs_folder, "physician_prediction_availability_characteristics.csv"
            ),
            keep_default_na=False,
        ),
        pd.read_csv(
            os.path.join(outputs_folder, "prediction_availability_denominators.csv")
        ),
    )
)
supplemental_tables.append(
    word_tables.WordTable(
        "eTable 5. Characteristics by physician prediction availability",
        availability_characteristics,
        header_groups=(
            ("VUmc", ("VUmc available", "VUmc missing")),
            ("LUMC", ("LUMC available", "LUMC missing")),
        ),
        header_labels=(
            "Characteristic",
            "Available",
            "Missing",
            "Available",
            "Missing",
        ),
        notes=(
            "Values are median [IQR] or n (%), as indicated. Comparisons are descriptive and are reported separately by hospital.",
        ),
    )
)
supplemental_tables = [
    replace(
        table,
        data=publication_table_frame(table.data),
        font_size=8,
        multiline_intervals=True,
        cohort_sections=True,
    )
    for table in supplemental_tables
]
for table in supplemental_tables:
    word_tables.write_word_tables(
        [table], os.path.join(formatted_tables_dir, f"{table.title}.docx")
    )

# Supplemental Digital Content.docx is a temporary workflow file: it is manually
# merged with the supplemental figures afterward, so it stays out of the tracked
# tables/ folder.
word_tables.write_word_tables(
    supplemental_tables,
    os.path.join(
        project_root,
        "results",
        "word_tables",
        report_cohort,
        "Supplemental Digital Content.docx",
    ),
)

word_tables.write_word_tables(
    low_risk_event_tables.local_diagnosis_review_word_tables(individual_entries),
    os.path.join(
        individual_case_dir, "Diagnosis timelines for local clinical review.docx"
    ),
    landscape=True,
)